# Precise Prefix Cache Routing

The optimized baseline uses **approximate (heuristic) prefix cache scoring**,
where the EPP estimates which replica is most likely to have a given prefix
cached based on a rolling hash of recent request prefixes. This works well
but can miss cache entries that were evicted and reloaded, or misjudge
cache state during bursty traffic.

**Precise prefix cache routing** replaces the heuristic with an event-driven
approach. Each vLLM instance publishes KV-cache events (inserts, evictions)
over ZMQ, and the EPP maintains a real-time index of exactly which blocks
are cached on which replica. This eliminates guesswork and maximizes cache
hit rates.

By the end of this notebook, you will:
- Understand the difference between approximate and precise cache scoring
- Deploy the precise prefix cache guide from the production stack
- Configure the KV-cache indexer with ZMQ events
- Benchmark cache hit rates and TTFT against the approximate baseline
- Monitor the indexer's event ingestion rate

In [ ]:
# Environment setup
import os

os.environ["NAMESPACE"] = "llm-d"
os.environ["MODEL_NAME"] = "Qwen/Qwen3-32B"

print("Namespace:", os.environ["NAMESPACE"])
print("Model:", os.environ["MODEL_NAME"])

## Approximate vs. Precise: How They Work

### Approximate (Heuristic) Scoring

The heuristic scorer maintains a Bloom filter or rolling hash per replica.
When a request arrives, the EPP hashes the prompt prefix and checks which
replica's filter matches. This is fast but has two failure modes:

1. **False positives** - The filter says a prefix is cached, but it was
   evicted. The request gets routed to a replica with no cache hit.
2. **Stale state** - After a burst of traffic, the filter lags behind
   actual cache contents, leading to suboptimal routing.

### Precise (Event-Driven) Scoring

With precise scoring, vLLM publishes ZMQ events for every KV-cache
block insertion and eviction. The EPP's KV-cache indexer consumes
these events and maintains an exact map:

```
block_hash -> {replica_1, replica_3}   (block is cached on replicas 1 and 3)
```

When a request arrives, the EPP tokenizes the prompt, computes block
hashes (using a configurable block size, typically 64 tokens), and
looks up exactly which replica has the most matching blocks.

The tradeoff is a small amount of network overhead from the ZMQ event
stream, but the routing accuracy improvement typically outweighs this.

In [ ]:
# Deploy the precise prefix cache guide
# This kustomize overlay configures both the vLLM model server
# (to emit ZMQ cache events) and the EPP (to consume them)

!kubectl apply -k llm-d-production-stack/guides/precise-prefix-cache-routing/model-server/ \
    -n $NAMESPACE

print("\nModel server deployed with ZMQ cache event publishing.")
print("Waiting for pods to be ready...")
!kubectl wait --for=condition=ready pod -l app=vllm -n $NAMESPACE --timeout=600s
print("Model server pods are ready.")

In [ ]:
# Deploy the router with the KV-cache indexer enabled

!kubectl apply -k llm-d-production-stack/guides/precise-prefix-cache-routing/router/ \
    -n $NAMESPACE

print("\nRouter deployed with precise prefix cache scorer.")
!kubectl wait --for=condition=ready pod -l app=epp -n $NAMESPACE --timeout=120s
print("EPP pods are ready.")

## Understanding the EPP Configuration

The precise prefix cache scorer is configured in the EPP's config with
a few key parameters:

- **`tokenProcessorConfig.blockSize`**: The number of tokens per cache block.
  Must match vLLM's block size (default: 64). The EPP tokenizes the prompt,
  splits it into blocks of this size, and hashes each block to look up
  cache state.

- **`zmqSubscriberConfig`**: Connection details for the ZMQ event stream
  from each vLLM instance. The EPP subscribes to cache insert/evict events.

- **`scorerWeight`**: How heavily to weight cache affinity vs. other signals
  (like load). A higher weight prioritizes cache hits over load balancing.

In [ ]:
# Inspect the EPP configuration to see the precise cache scorer settings

!kubectl get configmap epp-config -n $NAMESPACE -o yaml | grep -A 30 'prefix-cache'

print("\n--- Key configuration parameters ---")
print("tokenProcessorConfig.blockSize = 64")
print("  This must match vLLM's block_size setting.")
print("  Each prompt is split into 64-token blocks for cache lookup.")
print()
print("The scorer computes a cache affinity score for each replica:")
print("  score = (matched_blocks / total_blocks) * scorer_weight")
print("  This score is combined with load-awareness to pick the best replica.")

In [ ]:
# Run a multi-turn workload benchmark
# Multi-turn conversations are the best use case for prefix caching,
# because each turn reuses the entire conversation history as a prefix.

import subprocess
import json
import time

GATEWAY_IP = subprocess.check_output(
    "kubectl get svc envoy-gateway -n llm-d "
    "-o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True
).decode().strip().strip("'")

def chat_completion(messages, max_tokens=100):
    """Send a chat completion request and return the response with timing."""
    payload = json.dumps({
        "model": "Qwen/Qwen3-32B",
        "messages": messages,
        "max_tokens": max_tokens,
    })
    start = time.time()
    result = subprocess.run(
        ["curl", "-s", "-w", "\n%{time_starttransfer}",
         f"http://{GATEWAY_IP}:8080/v1/chat/completions",
         "-H", "Content-Type: application/json", "-d", payload],
        capture_output=True, text=True
    )
    total_time = time.time() - start
    lines = result.stdout.strip().split("\n")
    ttft = float(lines[-1]) if len(lines) > 1 else total_time
    return {"ttft": ttft, "total": total_time}

# Simulate a 5-turn conversation
conversation = [
    {"role": "user", "content": "I want to learn about distributed systems. "
     "Start by explaining what a distributed system is."},
]

print("Multi-turn conversation benchmark (5 turns)")
print("Each turn reuses the full conversation history as a prefix.")
print()

assistant_replies = [
    "Tell me about consensus algorithms like Raft and Paxos.",
    "How does replication work in distributed databases?",
    "What is the CAP theorem and why does it matter?",
    "Summarize everything we discussed in three bullet points.",
]

for turn in range(5):
    timing = chat_completion(conversation, max_tokens=150)
    prefix_tokens = sum(len(m["content"].split()) for m in conversation)
    print(f"Turn {turn + 1}: TTFT={timing['ttft']*1000:.0f}ms, "
          f"Total={timing['total']*1000:.0f}ms, "
          f"Prefix=~{prefix_tokens} words")

    # Add a mock assistant reply and the next user message
    conversation.append({"role": "assistant", "content": "[assistant response]"})
    if turn < len(assistant_replies):
        conversation.append({"role": "user", "content": assistant_replies[turn]})

print()
print("With precise prefix caching, TTFT should decrease on turns 2-5")
print("because the growing conversation prefix is found in the cache.")

In [ ]:
# Compare cache hit rates: precise vs. approximate
# Query the EPP metrics to see the actual cache hit ratio

print("=== Precise Prefix Cache Metrics ===")
print()

# Cache hit ratio
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=avg(epp_prefix_cache_hit_ratio{namespace="llm-d"})' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    val = float(r['value'][1])
    print(f'  Cache hit rate: {val:.2%}')
"

print()

# TTFT comparison
print("TTFT percentiles:")
for pct in [50, 90, 99]:
    quantile = pct / 100
    !kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
        wget -qO- "http://localhost:9090/api/v1/query?query=histogram_quantile({quantile}, rate(epp_request_ttft_seconds_bucket{{namespace=\"llm-d\"}}[5m]))" \
        2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    val = float(r['value'][1]) * 1000
    print(f'  p{pct}: {val:.0f} ms')
" 2>/dev/null || echo "  p$pct: (no data yet)"

print()
print("Typical improvement over approximate baseline:")
print("  Cache hit rate: +10-25% higher with precise scoring")
print("  TTFT p50:       20-40% lower due to more cache hits")
print("  TTFT p99:       15-30% lower (fewer cold-cache outliers)")

In [ ]:
# Monitor the KV-cache indexer's event ingestion rate
# This shows how fast the indexer is processing ZMQ events from vLLM

print("=== KV-Cache Indexer Health ===")
print()

# Check indexer event processing rate
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=rate(epp_indexer_events_total{namespace="llm-d"}[1m])' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    event_type = r['metric'].get('type', 'unknown')
    rate = float(r['value'][1])
    print(f'  {event_type} events/sec: {rate:.1f}')
"

print()

# Check the number of indexed blocks
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=epp_indexer_blocks_total{namespace="llm-d"}' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    pod = r['metric'].get('pod', 'unknown')
    blocks = r['value'][1]
    print(f'  {pod}: {blocks} blocks indexed')
"

print()
print("A healthy indexer should show:")
print("  - insert events/sec proportional to request throughput")
print("  - evict events/sec proportional to cache pressure")
print("  - Block count growing until cache reaches capacity, then stabilizing")
print()
print("If event processing lags behind (high event queue depth),")
print("consider increasing the indexer's CPU allocation.")

## Summary

Precise prefix cache routing gives the EPP an exact view of which KV-cache
blocks exist on which replica, eliminating the false positives and stale
state issues of heuristic scoring.

### When to Use Precise vs. Approximate

| Scenario | Recommendation |
|----------|----------------|
| Simple workloads, few replicas | Approximate is fine |
| Multi-turn conversations | Precise (biggest TTFT improvement) |
| High replica count (>8) | Precise (heuristic accuracy degrades with scale) |
| RAG with shared document prefixes | Precise (maximizes cross-request cache reuse) |

### Key Configuration

- `tokenProcessorConfig.blockSize = 64` (must match vLLM's block_size)
- ZMQ event stream from each vLLM instance to the EPP indexer
- Deploy using `guides/precise-prefix-cache-routing` in the production stack

### Next Steps

- **Tiered Prefix Cache**: Extend caching to CPU DRAM and NVMe disk for
  even higher effective cache capacity.
- **Predicted Latency Routing**: Combine cache-aware routing with latency
  prediction for SLO-driven request placement.